# Sentiment140 text preprocessing

This notebook prepares the Sentiment140 dataset for subsequent text representation and classification experiments.

The preprocessing workflow includes:

1. loading the cleaned dataset from Google Drive;
2. normalizing raw text;
3. removing URLs, HTML tags, and user mentions;
4. tokenizing and lemmatizing tweets with spaCy;
5. removing stopwords while preserving negations;
6. processing documents in batches;
7. validating the resulting corpus;
8. saving the final preprocessed dataset.

The output column, `texto_preprocesado`, contains the normalized textual representation used in later experiments.


In [1]:
# ==============================================================================
# SENTIMENT140 DATASET PREPROCESSING
# ==============================================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Environment setup and library installation

The required Python packages are installed and imported in this section. The English spaCy model `en_core_web_sm` is downloaded because Sentiment140 contains English-language tweets.

The parser and named-entity recognition components are not required for this preprocessing pipeline, which allows the workflow to focus on tokenization and lemmatization.


In [2]:
# ==============================================================================
# 1. INSTALL AND LOAD LIBRARIES
# ==============================================================================

!pip install spacy tqdm -q
!python -m spacy download en_core_web_sm -q

import re
import html
import time
import pandas as pd
import spacy

from pathlib import Path
from tqdm.auto import tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 79.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
# ==============================================================================
# 2. DEFINE PATHS
# ==============================================================================

BASE_DIR = Path("/content/drive/MyDrive/TFM/Dataset")

archivo_entrada = BASE_DIR / "sentiment140_limpio.csv"
archivo_salida = BASE_DIR / "sentiment140_preprocesado.csv"

print("Archivo de entrada:", archivo_entrada)
print("Archivo de salida:", archivo_salida)

Archivo de entrada: /content/drive/MyDrive/TFM/Dataset/sentiment140_limpio.csv
Archivo de salida: /content/drive/MyDrive/TFM/Dataset/sentiment140_preprocesado.csv


## 2. Dataset loading and initial inspection

The cleaned Sentiment140 dataset is loaded from Google Drive. Its dimensions, available columns, and first observations are inspected before applying any transformation.

This initial validation helps detect path errors, unexpected schemas, or missing columns at an early stage.


In [4]:
# ==============================================================================
# 3. LOAD THE DATASET
# ==============================================================================

df = pd.read_csv(archivo_entrada)

print("Dimensiones iniciales:", df.shape)
print("Columnas disponibles:", df.columns.tolist())

display(df.head())

Dimensiones iniciales: (1600000, 3)
Columnas disponibles: ['texto', 'label', 'sentimiento']


,texto,label,sentimiento
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,negativo
1,is upset that he can't update his Facebook by ...,0,negativo
2,@Kenichan I dived many times for the ball. Man...,0,negativo
3,my whole body feels itchy and like its on fire,0,negativo
4,"@nationwideclass no, it's not behaving at all....",0,negativo


In [5]:
# ==============================================================================
# 4. SELECT THE TEXT COLUMN
# ==============================================================================

COLUMNA_TEXTO = "texto"

if COLUMNA_TEXTO not in df.columns:
    raise ValueError(f"No se ha encontrado la columna '{COLUMNA_TEXTO}' en el dataset.")

df[COLUMNA_TEXTO] = df[COLUMNA_TEXTO].fillna("").astype(str)


## 3. spaCy configuration and negation handling

The spaCy language pipeline is configured for efficient preprocessing. Dependency parsing and named-entity recognition are disabled because they are not required for the current task.

Negations such as `not`, `never`, and `cannot` are deliberately preserved. This is especially important in sentiment analysis because removing a negation may reverse the meaning of a sentence.


In [6]:
# ==============================================================================
# 5. CONFIGURE SPACY
# ==============================================================================

# Disable the parser and NER components to speed up processing.
# Keep the lemmatizer enabled.
nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]
)

# spaCy stopwords
stopwords_spacy = set(nlp.Defaults.stop_words)

# Negations that will be explicitly preserved
negaciones = {
    "no", "not", "nor", "never", "none", "nobody", "nothing",
    "neither", "nowhere", "n't", "cannot",
    "can't", "won't", "don't", "doesn't", "didn't",
    "isn't", "aren't", "wasn't", "weren't",
    "haven't", "hasn't", "hadn't",
    "shouldn't", "wouldn't", "couldn't", "mustn't"
}

# Remove negations from the stopword set
stopwords_filtradas = stopwords_spacy - negaciones

print("Número de stopwords originales:", len(stopwords_spacy))
print("Número de stopwords tras conservar negaciones:", len(stopwords_filtradas))

Número de stopwords originales: 326
Número de stopwords tras conservar negaciones: 315


## 4. Initial text normalization

Before spaCy tokenization, each tweet undergoes a lightweight cleaning stage. HTML entities are decoded, text is converted to lowercase, and HTML tags, URLs, user mentions, and repeated whitespace are removed.

This step reduces noise while preserving the lexical information required for sentiment classification.


In [7]:
# ==============================================================================
# 6. AUXILIARY CLEANING FUNCTIONS
# ==============================================================================

def limpieza_inicial(texto):
    """
    Applies an initial cleaning stage before tokenization with spaCy.

    Operations:
    - Conversion to lowercase.
    - Decoding of HTML entities.
    - Removal of HTML tags.
    - Removal of URLs.
    - Removal of user mentions.
    - Whitespace normalization.
    """

    texto = str(texto)

    # Decode HTML entities, for example &amp;
    texto = html.unescape(texto)

    # Convert to lowercase
    texto = texto.lower()

    # Remove HTML tags
    texto = re.sub(r"<.*?>", " ", texto)

    # Remove URLs
    texto = re.sub(r"http\S+|www\.\S+", " ", texto)

    # Remove user mentions
    texto = re.sub(r"@\w+", " ", texto)

    # Normalize whitespace
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

## 5. Token-level preprocessing and lemmatization

The following function applies the main linguistic preprocessing rules to each spaCy document.

Punctuation, numerical tokens, non-alphabetic elements, and stopwords are removed, while meaningful negations are retained. Valid tokens are then converted to their lowercase lemmas to reduce inflectional variation.


In [8]:

# ==============================================================================
# 7. PREPROCESSING FUNCTION WITH SPACY
# ==============================================================================

def preprocesar_documento(doc):
    """
    Applies tokenization, removes numerical tokens, removes stopwords while
    preserving negations, and performs lemmatization.
    """

    tokens_limpios = []

    for token in doc:
        token_lower = token.text.lower().strip()

        # Remove whitespace tokens
        if token.is_space:
            continue

        # Remove punctuation unless it is a negation such as n't
        if token.is_punct and token_lower not in negaciones:
            continue

        # Remove numerical tokens
        if token.like_num or token.is_digit:
            continue

        # Remove non-alphabetic tokens unless they are negations
        if not token.is_alpha and token_lower not in negaciones:
            continue

        # Remove stopwords while preserving negations
        if token_lower in stopwords_filtradas:
            continue

        # Lemmatization
        lema = token.lemma_.lower().strip()

        # Additional validation for empty lemmas
        if lema == "" or lema == "-pron-":
            continue

        # Remove stopwords after lemmatization while preserving negations
        if lema in stopwords_filtradas:
            continue

        tokens_limpios.append(lema)

    return " ".join(tokens_limpios)

## 6. Batch preprocessing of the corpus

The complete corpus is processed with `nlp.pipe`, which is more efficient than applying the spaCy pipeline to one document at a time.

A progress bar reports the percentage completed, the percentage remaining, and the number of documents still pending. Execution time is recorded to evaluate the computational cost of preprocessing.


In [9]:
# ==============================================================================
# 8. APPLY BATCH PREPROCESSING
# ==============================================================================

inicio = time.time()

# Apply initial cleaning before spaCy processing
textos_limpios_iniciales = df[COLUMNA_TEXTO].apply(limpieza_inicial).tolist()

textos_preprocesados = []

batch_size = 1000
total_documentos = len(textos_limpios_iniciales)

# Progress bar
barra = tqdm(
    nlp.pipe(textos_limpios_iniciales, batch_size=batch_size),
    total=total_documentos,
    desc="Preprocesando documentos"
)

for i, doc in enumerate(barra, start=1):
    textos_preprocesados.append(preprocesar_documento(doc))

    # Percentage processed and remaining
    porcentaje_procesado = (i / total_documentos) * 100
    porcentaje_pendiente = 100 - porcentaje_procesado

    barra.set_postfix({
        "procesado": f"{porcentaje_procesado:.2f}%",
        "pendiente": f"{porcentaje_pendiente:.2f}%",
        "faltan": total_documentos - i
    })

df["texto_preprocesado"] = textos_preprocesados

fin = time.time()

print(f"Tiempo total de preprocesamiento: {fin - inicio:.2f} segundos")
print(f"Tiempo total de preprocesamiento: {(fin - inicio) / 60:.2f} minutos")


Preprocesando documentos:   0%|          | 0/1600000 [00:00<?, ?it/s]

Tiempo total de preprocesamiento: 4190.95 segundos
Tiempo total de preprocesamiento: 69.85 minutos


## 7. Removal of empty documents

Some tweets may become empty after URLs, mentions, punctuation, stopwords, and numerical tokens are removed.

These documents are excluded because they do not contain usable textual information for vectorization or model training. The number of removed observations is reported for traceability.


In [10]:
# ==============================================================================
# 9. REMOVE EMPTY DOCUMENTS AFTER CLEANING
# ==============================================================================

n_antes = len(df)

df["texto_preprocesado"] = df["texto_preprocesado"].fillna("").astype(str)
df = df[df["texto_preprocesado"].str.strip() != ""].copy()

n_despues = len(df)

print("Documentos antes de eliminar vacíos:", n_antes)
print("Documentos después de eliminar vacíos:", n_despues)
print("Documentos eliminados:", n_antes - n_despues)


Documentos antes de eliminar vacíos: 1600000
Documentos después de eliminar vacíos: 1588200
Documentos eliminados: 11800


## 8. Qualitative validation of preprocessing

A random sample of original and preprocessed tweets is displayed to verify that the transformation behaves as expected.

This comparison is useful for checking whether relevant sentiment-bearing words and negations are preserved while noisy elements are removed.


In [11]:
# ==============================================================================
# 10. COMPARE ORIGINAL AND PREPROCESSED TEXT
# ==============================================================================

display(
    df[[COLUMNA_TEXTO, "texto_preprocesado", "label"]]
    .sample(10, random_state=42)
)


,texto,texto_preprocesado,label
206017,@lisarinna I feel like a thrown away used rag.,feel like throw away rag,0
1085761,ended up with a very nice evening UP Is aweso...,end nice evening awesome surprised consider re...,1
1495269,Hectic day tomorrow gotta get up early night ...,hectic day tomorrow early night tweet later,1
24546,@JasonStatham1 why do i have to be one of 735 ...,not follow,0
286687,@poynterlubz ..er one haha i'll fail it.,er haha fail,0
670215,@kat_n you wait hen! lol. a girl can dream.,wait hen lol girl dream,0
750657,@renbren I am lost. Please help me find a good...,lose help find good home,0
1156899,@roniyaniv To see you all relaxed and happy af...,relaxed happy day hard exercise push balance t...,1
337578,@xoxofabuluz speak korean ? wow you're amazing...,speak korean wow amazing speak english indonesian,0
439568,Want to see NFG again!,want nfg,0


## 9. Export of the preprocessed dataset

The resulting dataset is saved as a UTF-8 encoded CSV file. Keeping the processed corpus in a separate file avoids repeating the computationally expensive spaCy pipeline in later experiments.


In [12]:

# ==============================================================================
# 11. SAVE THE PREPROCESSED DATASET
# ==============================================================================

df.to_csv(archivo_salida, index=False, encoding="utf-8")

print("Dataset preprocesado guardado correctamente en:")
print(archivo_salida)

print("Dimensiones finales:", df.shape)


Dataset preprocesado guardado correctamente en:
/content/drive/MyDrive/TFM/Dataset/sentiment140_preprocesado.csv
Dimensiones finales: (1588200, 4)


## 10. Reloading and structural validation

The saved dataset is reloaded to verify that it can be read correctly and that its structure has been preserved.

The following cells inspect its dimensions, column names, data types, and representative observations.


In [13]:
# ==============================================================================
# 12. LOAD THE PREPROCESSED DATASET LATER
# ==============================================================================

df_sentiment140_preprocesado = pd.read_csv(
    "/content/drive/MyDrive/TFM/Dataset/sentiment140_preprocesado.csv"
)

print(df_sentiment140_preprocesado.shape)
display(df_sentiment140_preprocesado.head())


print(df.shape)
print(df.head())
print(df.info())
print(df.columns)


(df["texto_preprocesado"].fillna("").str.strip() == "").sum()

# Exact duplicates
df.duplicated().sum()

# Duplicates in the text column
df["texto"].duplicated().sum()


df[df["texto"].duplicated(keep=False)].sort_values("texto").head(10)


(1588200, 4)


,texto,label,sentimiento,texto_preprocesado
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0,negativo,awww bummer shoulda david carr day d
1,is upset that he can't update his Facebook by ...,0,negativo,upset not update facebook texte cry result sch...
2,@Kenichan I dived many times for the ball. Man...,0,negativo,dive time ball manage save rest bound
3,my whole body feels itchy and like its on fire,0,negativo,body feel itchy like fire
4,"@nationwideclass no, it's not behaving at all....",0,negativo,no not behave mad not


In [14]:
print(df.shape )
print(df.head())
print(df.info())
print(df.columns)

(1588200, 4)
                                               texto  label sentimiento  \
0  @switchfoot http://twitpic.com/2y1zl - Awww, t...      0    negativo   
1  is upset that he can't update his Facebook by ...      0    negativo   
2  @Kenichan I dived many times for the ball. Man...      0    negativo   
3    my whole body feels itchy and like its on fire       0    negativo   
4  @nationwideclass no, it's not behaving at all....      0    negativo   

                                  texto_preprocesado  
0               awww bummer shoulda david carr day d  
1  upset not update facebook texte cry result sch...  
2              dive time ball manage save rest bound  
3                          body feel itchy like fire  
4                              no not behave mad not  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1588200 entries, 0 to 1588199
Data columns (total 4 columns):
 #   Column              Non-Null Count    Dtype 
---  ------              --------------    -

## 11. Class-distribution analysis

The absolute and relative frequencies of the sentiment labels are examined to confirm whether the dataset remains balanced after preprocessing and the removal of empty documents.

Class balance is important because substantial differences in class frequency may affect the interpretation of classification metrics.


In [15]:
df["label"].value_counts()
df["label"].value_counts(normalize=True) * 100

,proportion
label,
0,50.05629
1,49.94371


## 12. Final data-quality checks

The remaining cells verify that the preprocessed text column contains no empty strings and examine exact or text-level duplicates.

Duplicate analysis helps identify repeated tweets that could introduce redundancy or information leakage if equivalent observations appear in both training and test sets.


In [17]:
(df["texto_preprocesado"].fillna("").str.strip() == "").sum()

np.int64(2)

In [19]:
# Duplicados exactos
df.duplicated().sum()

# Duplicados en la columna textual
df["texto"].duplicated().sum()

np.int64(18225)

In [21]:
df[df["texto"].duplicated(keep=False)].sort_values("texto").head(10)

,texto,label,sentimiento,texto_preprocesado
269732,David must be hospitalized for five days end...,0,negativo,david hospitalize day end july palatine tonsil...
269765,David must be hospitalized for five days end...,0,negativo,david hospitalize day end july palatine tonsil...
1124196,bathroom is clean..... now on to more enjoya...,1,positivo,bathroom clean enjoyable task
1124194,bathroom is clean..... now on to more enjoya...,1,positivo,bathroom clean enjoyable task
536784,#IMISSCATH #IMISSCATH #IMISSCATH #IMISSCATH #...,0,negativo,imisscath imisscath imisscath imisscath imissc...
536771,#IMISSCATH #IMISSCATH #IMISSCATH #IMISSCATH #...,0,negativo,imisscath imisscath imisscath imisscath imissc...
1347335,#seb-day #seb-day #seb-day #seb-day #seb-day ...,1,positivo,seb day seb day seb day seb day seb day seb da...
1355095,#seb-day #seb-day #seb-day #seb-day #seb-day ...,1,positivo,seb day seb day seb day seb day seb day seb da...
328982,*tear*,0,negativo,tear
285397,*tear*,0,negativo,tear


In [23]:
df[df["texto"].duplicated(keep=False)].sort_values("texto").head(10)

,texto,label,sentimiento,texto_preprocesado
269732,David must be hospitalized for five days end...,0,negativo,david hospitalize day end july palatine tonsil...
269765,David must be hospitalized for five days end...,0,negativo,david hospitalize day end july palatine tonsil...
1124196,bathroom is clean..... now on to more enjoya...,1,positivo,bathroom clean enjoyable task
1124194,bathroom is clean..... now on to more enjoya...,1,positivo,bathroom clean enjoyable task
536784,#IMISSCATH #IMISSCATH #IMISSCATH #IMISSCATH #...,0,negativo,imisscath imisscath imisscath imisscath imissc...
536771,#IMISSCATH #IMISSCATH #IMISSCATH #IMISSCATH #...,0,negativo,imisscath imisscath imisscath imisscath imissc...
1347335,#seb-day #seb-day #seb-day #seb-day #seb-day ...,1,positivo,seb day seb day seb day seb day seb day seb da...
1355095,#seb-day #seb-day #seb-day #seb-day #seb-day ...,1,positivo,seb day seb day seb day seb day seb day seb da...
328982,*tear*,0,negativo,tear
285397,*tear*,0,negativo,tear
